In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# GPT VISION TOP-5 LABELING
# ============================================================

import os
import re
import json
import time
import base64
import mimetypes
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


# ============================================================
# CONFIGURATION
# ============================================================

# Your image folder
IMAGE_DIR = r"google_test_images/SMER_ViT"

# Your template CSV
INPUT_CSV = r"data/Labels_Top5.csv"

# Output CSV
OUTPUT_CSV = r"data/Labels_Top5_GPT_Adv_SMER.csv"

# Put your API key here OR use environment variable OPENAI_API_KEY
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
# Use a vision-capable model available in your account
MODEL = "gpt-5.6"

# Retry settings
MAX_RETRIES = 5
SLEEP_BETWEEN_REQUESTS = 0.3


# ============================================================
# CLIENT
# ============================================================

client = OpenAI(api_key=OPENAI_API_KEY)


# ============================================================
# PROMPT
# ============================================================

PROMPT = """
Examine this image carefully.

Identify the main object shown in the image.

Give exactly five most likely object categories, ordered from
most likely to least likely.

Rules:
- Give exactly 5 guesses.
- Use short common object/category names.
- Do not explain your reasoning.
- Do not add any text outside the JSON.
- Do not use numbering.
- Return valid JSON exactly in this format:

{
  "top5": [
    "label1",
    "label2",
    "label3",
    "label4",
    "label5"
  ]
}
"""


# ============================================================
# IMAGE → BASE64 DATA URL
# ============================================================

def image_to_data_url(image_path):

    mime_type, _ = mimetypes.guess_type(image_path)

    if mime_type is None:
        mime_type = "image/png"

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    encoded = base64.b64encode(image_bytes).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


# ============================================================
# PARSE MODEL OUTPUT
# ============================================================

def parse_top5(text):

    if text is None:
        return None

    text = text.strip()

    # Remove Markdown fences if the model unexpectedly adds them
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # --------------------------------------------------------
    # First try: normal JSON
    # --------------------------------------------------------

    try:

        data = json.loads(text)

        labels = data.get("top5", [])

        if isinstance(labels, list) and len(labels) >= 5:

            labels = [
                str(x).strip()
                for x in labels[:5]
            ]

            return labels

    except Exception:
        pass

    # --------------------------------------------------------
    # Fallback: extract quoted strings
    # --------------------------------------------------------

    quoted = re.findall(r'"([^"]+)"', text)

    # remove "top5" key if extracted
    quoted = [
        x for x in quoted
        if x.lower() != "top5"
    ]

    if len(quoted) >= 5:
        return quoted[:5]

    return None


# ============================================================
# QUERY GPT
# ============================================================

def query_gpt(image_path):

    image_url = image_to_data_url(image_path)

    response = client.responses.create(

        model=MODEL,

        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": PROMPT
                    },
                    {
                        "type": "input_image",
                        "image_url": image_url,
                        "detail": "high"
                    }
                ]
            }
        ]
    )

    text = response.output_text

    labels = parse_top5(text)

    return labels, text


# ============================================================
# LOAD CSV
# ============================================================

# Resume automatically if output already exists
if os.path.exists(OUTPUT_CSV):

    print("Existing output found.")
    print("Resuming from:", OUTPUT_CSV)

    df = pd.read_csv(OUTPUT_CSV)

else:

    df = pd.read_csv(INPUT_CSV)


# ============================================================
# ENSURE REQUIRED COLUMNS EXIST
# ============================================================

label_columns = [
    "Label 1",
    "Label 2",
    "Label 3",
    "Label 4",
    "Label 5"
]

for col in label_columns:

    if col not in df.columns:
        df[col] = ""


# Optional logging columns
if "status" not in df.columns:
    df["status"] = ""

if "raw_response" not in df.columns:
    df["raw_response"] = ""


print("\nTotal images:", len(df))
print("Image directory:", IMAGE_DIR)
print("Model:", MODEL)
print()


# ============================================================
# MAIN LOOP
# ============================================================

for idx in tqdm(range(len(df)), desc="GPT labeling"):

    image_name = str(df.loc[idx, "image_name"])

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    # --------------------------------------------------------
    # SKIP ALREADY COMPLETED IMAGES
    # --------------------------------------------------------

    already_done = all(
        pd.notna(df.loc[idx, col])
        and str(df.loc[idx, col]).strip() != ""
        for col in label_columns
    )

    if already_done:
        continue


    # --------------------------------------------------------
    # IMAGE NOT FOUND
    # --------------------------------------------------------

    if not os.path.exists(image_path):

        print("\nImage not found:", image_path)

        df.loc[idx, "status"] = "image_not_found"

        df.to_csv(
            OUTPUT_CSV,
            index=False
        )

        continue


    # --------------------------------------------------------
    # QUERY WITH RETRIES
    # --------------------------------------------------------

    success = False

    for attempt in range(MAX_RETRIES):

        try:

            labels, raw_response = query_gpt(
                image_path
            )

            # Store raw model answer
            df.loc[idx, "raw_response"] = raw_response


            if labels is None or len(labels) != 5:

                raise ValueError(
                    f"Could not parse 5 labels: {raw_response}"
                )


            # ------------------------------------------------
            # WRITE LABELS
            # ------------------------------------------------

            for j in range(5):

                df.loc[
                    idx,
                    f"Label {j+1}"
                ] = labels[j]


            df.loc[idx, "status"] = "success"

            success = True

            break


        except Exception as e:

            print(
                f"\nError: {image_name} "
                f"(attempt {attempt+1}/{MAX_RETRIES})"
            )

            print(e)

            df.loc[idx, "status"] = (
                f"failed_attempt_{attempt+1}"
            )

            time.sleep(
                2 ** attempt
            )


    # --------------------------------------------------------
    # SAVE AFTER EVERY IMAGE
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_CSV,
        index=False
    )


    if not success:

        print(
            "\nFAILED permanently:",
            image_name
        )


    time.sleep(
        SLEEP_BETWEEN_REQUESTS
    )


# ============================================================
# FINAL SAVE
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

success_count = (
    df["status"] == "success"
).sum()

failed_count = (
    df["status"] != "success"
).sum()


print("\n====================================")
print("FINISHED")
print("====================================")

print("Total:", len(df))
print("Successful:", success_count)
print("Failed / missing:", failed_count)

print("\nSaved to:")
print(OUTPUT_CSV)

In [ ]:
# ============================================================
# RETRY ONLY FAILED / INCOMPLETE GPT ROWS
# ============================================================

import os
import time
import pandas as pd
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

OUTPUT_CSV = r"data/Labels_Top5_GPT_Adv.csv"

MAX_RETRIES = 8

label_columns = [
    "Label 1",
    "Label 2",
    "Label 3",
    "Label 4",
    "Label 5"
]


# ============================================================
# LOAD EXISTING RESULTS
# ============================================================

df = pd.read_csv(OUTPUT_CSV)

print("Total rows:", len(df))


# ============================================================
# CHECK WHETHER ROW IS COMPLETE
# ============================================================

def row_is_complete(row):

    return all(
        pd.notna(row[col])
        and str(row[col]).strip() != ""
        for col in label_columns
    )


# ============================================================
# FIND ONLY INCOMPLETE ROWS
# ============================================================

incomplete_indices = [
    idx
    for idx, row in df.iterrows()
    if not row_is_complete(row)
]

print("Already complete:", len(df) - len(incomplete_indices))
print("Need retry:", len(incomplete_indices))


# ============================================================
# RETRY INCOMPLETE IMAGES
# ============================================================

for idx in tqdm(
    incomplete_indices,
    desc="Retrying incomplete GPT images"
):

    image_name = str(
        df.loc[idx, "image_name"]
    )

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )


    # --------------------------------------------------------
    # FILE CHECK
    # --------------------------------------------------------

    if not os.path.exists(image_path):

        print(
            "\nImage not found:",
            image_path
        )

        df.loc[
            idx,
            "status"
        ] = "image_not_found"

        df.to_csv(
            OUTPUT_CSV,
            index=False
        )

        continue


    # --------------------------------------------------------
    # RETRY
    # --------------------------------------------------------

    success = False

    for attempt in range(MAX_RETRIES):

        try:

            labels, raw_response = query_gpt(
                image_path
            )


            if labels is None or len(labels) != 5:

                raise ValueError(
                    f"Could not parse 5 labels: {raw_response}"
                )


            # -----------------------------------------------
            # WRITE LABELS
            # -----------------------------------------------

            for j in range(5):

                df.loc[
                    idx,
                    f"Label {j+1}"
                ] = labels[j]


            df.loc[
                idx,
                "raw_response"
            ] = raw_response


            df.loc[
                idx,
                "status"
            ] = "success"


            success = True

            print(
                f"Recovered: {image_name}"
            )

            break


        except Exception as e:

            error_text = str(e)

            print(
                f"\nError on {image_name}"
            )

            print(
                f"Attempt {attempt + 1}/{MAX_RETRIES}"
            )

            print(error_text)


            # -----------------------------------------------
            # BILLING / QUOTA ERROR
            # -----------------------------------------------

            if (
                "insufficient_quota" in error_text.lower()
                or "billing" in error_text.lower()
                or "quota" in error_text.lower()
            ):

                print(
                    "\nBilling/quota problem detected."
                )

                df.loc[
                    idx,
                    "status"
                ] = "quota_error"

                # save before stopping
                df.to_csv(
                    OUTPUT_CSV,
                    index=False
                )

                raise RuntimeError(
                    "OpenAI quota/billing issue. "
                    "Results saved safely. "
                    "Restore API access and rerun this cell."
                )


            # -----------------------------------------------
            # TEMPORARY ERRORS
            # -----------------------------------------------

            if (
                "429" in error_text
                or "500" in error_text
                or "502" in error_text
                or "503" in error_text
                or "504" in error_text
                or "timeout" in error_text.lower()
                or "rate limit" in error_text.lower()
            ):

                delay = min(
                    60,
                    2 ** attempt
                )

                print(
                    f"Temporary error. Retrying in {delay}s..."
                )

                time.sleep(delay)

            else:

                # Unknown error: don't endlessly retry
                print(
                    "Non-temporary error."
                )

                break


    # --------------------------------------------------------
    # SAVE AFTER EVERY IMAGE
    # --------------------------------------------------------

    if not success:

        df.loc[
            idx,
            "status"
        ] = "still_failed"


    df.to_csv(
        OUTPUT_CSV,
        index=False
    )


# ============================================================
# FINAL SAVE
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

complete_count = sum(
    row_is_complete(row)
    for _, row in df.iterrows()
)

remaining_count = (
    len(df) - complete_count
)


print("\n====================================")
print("GPT RETRY FINISHED")
print("====================================")

print("Total:", len(df))
print("Complete:", complete_count)
print("Still incomplete:", remaining_count)

print("\nSaved:")
print(OUTPUT_CSV)